# Decile Testing

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Global Variables and Data

In [2]:
note_path = os.getcwd()
repo_path = os.path.abspath(os.path.join(note_path, ".."))
src_path  = os.path.join(repo_path, "src")
data_path = os.path.join(repo_path, "data")

In [3]:
sys.path.append(src_path)
from QuantileFitter import QuantileFitter

In [4]:
trend_path = os.path.join(data_path, "GenericTrendSignal.parquet")
df_data    = (pd
    .read_parquet(path = trend_path, engine = "pyarrow")
    .assign(signal_name = lambda x: x.Ticker + "_" + x.window.astype(str)))

In [5]:
print(src_path)

A:\GitHub\SignalTrader\src


# Decile Models

In [6]:
df_data.head(3)

,Ticker,Date,lag_signal,signal,window,rtn,signal_name
0,AGG,2003-10-01,NaN,-0.121794,8,-0.000486,AGG_8
1,AGG,2003-10-02,-0.121794,-0.177490,8,-0.001559,AGG_8
2,AGG,2003-10-03,-0.177490,-0.403561,8,-0.007219,AGG_8


In [7]:
quintile_fitter = (QuantileFitter(
    signal_col = "lag_signal",
    rtn_col    = "rtn", 
    group_col  = "signal_name",
    date_col   = "Date"))

TypeError: QuantileFitter.__init__() got an unexpected keyword argument 'signal_col'

In [ ]:
df_tmp = quintile_fitter.fit_quantile_full_sample(df_data)

In [ ]:
df_tmp.keys()

In [ ]:
(df_tmp
    ["data"]
    .assign(signal_rtn = lambda x: np.where(x.signal_rtn == 0, np.nan, x.signal_rtn))
    .dropna()
    .pivot(index = "Date", columns = "signal_name", values = "signal_rtn")
    .apply(lambda x: x * (0.1 / (x.ewm(span = 100, adjust = False).std() * np.sqrt(252))))
    .mean(axis = 1).to_frame(name = "tmp")
    .apply(lambda x: np.cumprod(1 + x))
    .plot(logy = True, legend = False))